# 05 · Matriz de situaciones

**Qué hace.** Para cada situación de juego cuenta tres cosas:
- **Qué hace el América de Jardine:** índice Proactivo–Reactivo, PPDA, altura de recuperación y field tilt.
- **Qué le funciona:** xG a favor y en contra por 90 minutos, con intervalo por bootstrap de partidos.
- **En qué se diferencia de Solari y Ortiz** en esa misma situación.

Las situaciones son con uno más, con uno menos, ganando por 1 o por 2+, empatando en cada tiempo y perdiendo por 1 o por 2+ (`ojo.lentes.situaciones`). Es la base de la matriz y de las recomendaciones de la app.

**Qué necesita.** Pasos 02–05 corridos (`data/processed/` y `outputs/panel/`).

**Cómo probarlo.** En VS Code, kernel `.venv`, *Run All*. Tarda menos de un minuto. La tabla se guarda en `outputs/hallazgos/matriz.parquet` (local, no se sube).

> IMPOR1: sin el Clausura 2026 (`INCLUIR_PRUEBA = False`).
>
> IMPOR2: solo agregados; el notebook se sube sin salidas.

In [ ]:
import numpy as np
import pandas as pd

from ojo.config import DIR_SALIDAS, params
from ojo.datos import almacen
from ojo.framework import partidos as fp
from ojo.lentes import defensa, hallazgo, indice, minuto, situaciones

P = params()
EQUIPO = P["equipo"]["team_id"]
RNG = np.random.default_rng(P["simulacion"]["semilla"])
INCLUIR_PRUEBA = False  # IMPOR1
DIR_HALLAZGOS = DIR_SALIDAS / "hallazgos"
DIR_HALLAZGOS.mkdir(parents=True, exist_ok=True)

ENTRENADORES = {d["manager_id"]: d["nombre"].split()[-1] for d in P["predecesores"]}
ENTRENADORES[P["entrenador"]["manager_id"]] = P["entrenador"]["nombre"].split()[-1]
NUEVO, PREVIOS = list(ENTRENADORES.values())[-1], "Predecesores"

partidos = fp.con_torneo(fp.del_equipo(fp.validos(almacen.cargar("partidos")), EQUIPO))
partidos = partidos[partidos["entrenador_id"].isin(ENTRENADORES)].copy()
partidos["prueba"] = fp.es_prueba(partidos)
if not INCLUIR_PRUEBA:
    partidos = partidos[~partidos["prueba"]]
partidos["grupo"] = np.where(partidos["entrenador_id"].map(ENTRENADORES) == NUEVO, NUEVO, PREVIOS)
ids = partidos["match_id"].astype(int).tolist()
eventos = almacen.cargar("eventos", ids, columnas=[
    "match_id", "indice", "periodo", "minuto", "equipo_id", "tipo", "x", "y", "fin_x", "fin_y", "patron", "pase_tipo",
    "duelo_tipo", "recuperacion_fallida", "intercepcion_resultado", "tiro_resultado", "tarjeta", "duracion",
    "es_transicion", "directness"])
panel = minuto.cargar_panel("minuto").query("match_id in @ids")
print(f"{len(ids)} partidos · {panel[['match_id', 'periodo', 'minuto']].drop_duplicates().shape[0]:,} minutos")

## 1. Una fila por minuto, a dos bandas

Sumas del panel (xG, pases al último tercio), de `ojo.lentes.defensa` (PPDA y recuperaciones) y el índice de cada equipo en ese minuto. A cada minuto se le pone su situación.

In [ ]:
CLAVES = ["match_id", "periodo", "minuto"]
lados = panel.pivot_table(index=CLAVES, columns="lado", values=["xg", "pases_ultimo_tercio"], aggfunc="sum", fill_value=0)
t = pd.DataFrame({"xg_favor": lados[("xg", minuto.PROPIO)], "xg_contra": lados[("xg", minuto.RIVAL)],
                  "tercio_favor": lados[("pases_ultimo_tercio", minuto.PROPIO)],
                  "tercio_contra": lados[("pases_ultimo_tercio", minuto.RIVAL)]})

d = defensa.piezas_por_minuto(eventos)
propio, rival = d[d["equipo_id"] == EQUIPO].set_index(CLAVES), d[d["equipo_id"] != EQUIPO].set_index(CLAVES)
t = t.join(pd.DataFrame({"acciones_def": propio["acciones_def_zona"], "rec_n": propio["recuperaciones_n"],
                         "rec_x": propio["recuperaciones_x_suma"], "pases_rival_zona": rival["pases_zona_propia"]}), how="outer")
t[situaciones.SUMAS] = t[situaciones.SUMAS].fillna(0.0)

comp = indice.componentes(panel, indice.piezas_por_minuto(eventos))
tab_ind = indice.calcular(comp, indice.ajustar(comp))  # escala: estos mismos minutos de entrenamiento
ind = tab_ind.pivot_table(index=CLAVES, columns="lado", values="indice")
t = t.join(ind.rename(columns={minuto.PROPIO: "indice_favor", minuto.RIVAL: "indice_rival"}))

estado = situaciones.por_minuto(eventos, EQUIPO).set_index(CLAVES)
t = t.join(estado).dropna(subset=["dif_goles"]).reset_index()
t["situacion"] = situaciones.asignar(t)
t = t.merge(partidos[["match_id", "grupo", "torneo_corto"]], on="match_id")
t["xg_dif"] = t["xg_favor"] - t["xg_contra"]
t.groupby(["grupo", "situacion"]).size().unstack("grupo").reindex(list(situaciones.SITUACIONES))

## 2. La matriz

Para Jardine en cada situación: qué hace (índice, PPDA, altura de recuperación, field tilt) y qué le funciona (xG por 90 a favor, en contra y la diferencia, con IC 95 % por bootstrap de partidos). Al lado, lo mismo con Solari y Ortiz y la diferencia en el xG neto, con p y FDR entre situaciones.

In [ ]:
def ic_dif(datos: pd.DataFrame) -> tuple[float, float]:
    medias = hallazgo.medias_bootstrap(datos, "xg_dif", RNG)
    bajo, alto = hallazgo.intervalo(medias)
    return 90 * bajo, 90 * alto


filas = []
for sit, nombre in situaciones.SITUACIONES.items():
    sub = t[t["situacion"] == sit]
    j, p_ = sub[sub["grupo"] == NUEVO], sub[sub["grupo"] == PREVIOS]
    if j.empty:
        continue
    r = situaciones.resumir(j, ["situacion"]).iloc[0].to_dict()
    r["xg_dif_bajo"], r["xg_dif_alto"] = ic_dif(j)
    if p_["match_id"].nunique() >= 2:
        rp = situaciones.resumir(p_, ["situacion"]).iloc[0]
        medias = hallazgo.medias_bootstrap(j, "xg_dif", RNG) - hallazgo.medias_bootstrap(p_, "xg_dif", RNG)
        bajo, alto = hallazgo.intervalo(medias)
        r.update({"previos_xg_dif_90": rp["xg_dif_90"], "previos_indice": rp["indice"], "previos_ppda": rp["ppda"],
                  "previos_minutos": rp["minutos"], "contra_previos": r["xg_dif_90"] - rp["xg_dif_90"],
                  "contra_previos_bajo": 90 * bajo, "contra_previos_alto": 90 * alto, "p": hallazgo.p_bootstrap(medias)})
    r["nombre"] = nombre
    filas.append(r)
matriz = pd.DataFrame(filas)
matriz["q"] = hallazgo.fdr_bh(matriz["p"])
matriz["distinto_de_previos"] = (matriz["q"] <= P["hallazgo"]["fdr_q"]) & (np.sign(matriz["contra_previos_bajo"]) == np.sign(matriz["contra_previos_alto"]))
base = situaciones.resumir(t[t["grupo"] == NUEVO].assign(todo="todo"), ["todo"]).iloc[0]
matriz.to_parquet(DIR_HALLAZGOS / "matriz.parquet", index=False)
pd.Series(base).to_frame("Jardine, todos los minutos").T.to_parquet(DIR_HALLAZGOS / "matriz_base.parquet")
print(f"Base de Jardine: índice {base['indice']:.0f}, PPDA {base['ppda']:.1f}, xG neto {base['xg_dif_90']:+.2f} por 90")
matriz[["nombre", "minutos", "partidos", "indice", "indice_rival", "ppda", "altura_recuperacion", "field_tilt",
        "xg_favor_90", "xg_contra_90", "xg_dif_90", "xg_dif_bajo", "xg_dif_alto", "previos_xg_dif_90", "contra_previos", "q",
        "distinto_de_previos"]].round(2)

## Lectura

Corrida del 18 oct (110 partidos de Jardine y 82 de Solari y Ortiz, sin el CL26). Base de Jardine: índice 57, PPDA 16.2 y xG neto de +0.66 por 90.

1. **Empatando, propone y le funciona.**
    - Primer tiempo: índice 61 frente a 52 del rival, xG neto +0.50 (IC 0.25 a 0.75).
    - Segundo tiempo: +0.68 (0.34 a 1.09).
2. **Perdiendo por 1, se vuelca.**
    - Índice 66 frente a 32 del rival, su presión más alta (PPDA 12.6) y field tilt de 0.75.
    - xG neto +1.01 (0.35 a 1.78): remonta el juego, no solo el balón.
3. **Ganando por 1, se repliega y el partido se empareja.**
    - El índice baja a 49 (el rival sube a 55) y el PPDA sube a 17.4.
    - El xG neto cae a +0.31, con un IC que cruza 0 (−0.04 a 0.66).
    - Es la situación con más minutos en la que el América deja de mandar: la recomendación para el cuerpo técnico (hipótesis) es no regalar la iniciativa con 1-0.
4. **Ganando por 2 o más, administra.** Índice 46 y el rival tiene más el balón, pero el xG neto sigue positivo (+0.88; 0.04 a 1.68).
5. **Con uno más, domina.** Índice 65 frente a 34, PPDA 13.2 y xG neto +1.53 (0.70 a 2.51). Con uno menos hay solo 76 minutos (7 partidos): muestra chica.
6. **Frente a Solari y Ortiz, la matriz no se separa.**
    - En ninguna situación la diferencia de xG neto sobrevive al FDR (q = 0.99 en todas).
    - Lo que distingue a Jardine está en *cómo* juega (notebooks 02–04), no en cuánto saca de cada situación.
    - Es una cautela honesta para el reporte.

Las recomendaciones de la app salen de estos números con las reglas de `ojo.salidas.decision` (framework.md, "Matriz de situaciones") y se presentan como hipótesis.